#  Group Completion Analysis
This notebook analyzes performance differences based on total_correct scores.

In [15]:

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

metrics_df = pd.read_excel("expanded_combined_metrics_analysis.xlsx")
diagnostic_df = pd.read_excel("expanded_diagnostic_performance.xlsx")
kc_round_counts_df = pd.read_excel("updated_kc_round_counts.xlsx")
group_df = pd.read_excel("group_data.xlsx")


In [ ]:

# # Compute high vs. low performing groups
# group_avg = metrics_df.groupby('group_id')['total_correct'].mean()
# median_correct = group_avg.median()

# metrics_df['performance_group'] = metrics_df['group_id'].apply(lambda x: 'High' if group_avg[x] >= median_correct else 'Low')
# group_perf_map = metrics_df[['group_id', 'performance_group']].drop_duplicates()
# diagnostic_df = diagnostic_df.merge(group_perf_map, on='group_id', how='left')
# kc_round_counts_df = kc_round_counts_df.merge(group_perf_map, on='group_id', how='left')


### Compute teams that successfully learned individual KCs and as a result the entire domain

In [ ]:
# 

### Compute individuals that successfully learned individual KCs, initially and consistently

In [17]:
# For each user and KC, sort by round and track response over time
diagnostic_sorted = diagnostic_df.sort_values(['user_id', 'domain', 'kc_id', 'round'])


# Function to detect learning and unlearning
def analyze_learning(group):
    group = group.sort_values('round')
    if len(group) > 4:
        group = group.iloc[:4]
    group = group.reset_index(drop=True)
    
    learned = False
    learned_at_round = None
    learned_at_index = None
    unlearned = False
    
    N_rounds = len(group)

    final_ratio = group['optimal_response_ratio'].iloc[-1]
    # Determine if user learned the KC (final response ratio == 1)
    if final_ratio == 1.0:
        learned = True
        
        # When did they first reach 1.0?
        # learned_at_idx = group[group['optimal_response_ratio'] == 1.0].index[0]
        # learned_at_round = group.loc[learned_at_idx, 'round']
        first_learned_rows = group[group['optimal_response_ratio'] == 1.0]
        learned_at_index = group.index.get_loc(first_learned_rows.index[0]) + 1
        learned_at_round = first_learned_rows['round'].iloc[0]
        
        # Check if all later rounds (after learned_at_round) stayed at 1
        rounds_after = group[group['round'] >= learned_at_round]
        if not all(rounds_after['optimal_response_ratio'] == 1.0):
            unlearned = True
            final_ratio = rounds_after['optimal_response_ratio'].iloc[-1]  # final value if unlearned
    
    return pd.Series({
        'learned': learned,
        'learned_at_round': learned_at_index,
        'unlearned': unlearned,
        'final_optimal_response_ratio': final_ratio,
        'number_rounds': N_rounds
    })


# Apply the analysis function to each (user_id, kc_id)
learning_analysis_df = diagnostic_sorted.groupby(['user_id', 'group_id', 'domain', 'kc_id']).apply(analyze_learning).reset_index()
learning_analysis_df.head()

learning_analysis_df.to_excel('learning_analysis.xlsx')


In [22]:
# Update groups based on individuals/groups who learned

if not 'learning_analysis_df' in locals():
    learning_analysis_df = pd.read_excel('learning_analysis.xlsx')

metrics_df = pd.read_excel("expanded_combined_metrics_analysis.xlsx")

# ---- STEP 1: total_rounds per individual ----
# update the total_rounds in metrics_df based on 
grouped_rounds = learning_analysis_df.groupby(['user_id', 'domain'])['number_rounds'].sum().reset_index()

# ---- STEP 2: learning flags per individual ----
individuals_learned = learning_analysis_df.groupby(['user_id', 'domain'])['learned'].apply(all).reset_index(name='individuals_learned')
individuals_unlearned = learning_analysis_df.groupby(['user_id', 'domain'])['unlearned'].apply(any).reset_index(name='individuals_unlearned')

# Merge all individual-level info
individual_info = grouped_rounds.merge(individuals_learned, on=['user_id', 'domain']).merge(individuals_unlearned, on=['user_id', 'domain'])

# ---- STEP 3: update metrics_df (join by user_id and domain) ----
metrics_df = metrics_df.merge(individual_info, on=['user_id', 'domain'], how='left')

# ---- STEP 4: compute group-level summaries ----
# check if the individuals in the group all have 'TRUE' to learned and if anyone in group had 'TRUE' to unlearn
groups_learned = learning_analysis_df.groupby(['group_id', 'domain'])['learned'].apply(all).reset_index(name='groups_learned')
groups_unlearned = learning_analysis_df.groupby(['group_id', 'domain'])['unlearned'].apply(any).reset_index(name='groups_unlearned')

# metrics_df.set_index(['user_id', 'domain'], inplace=True)
# metrics_df['total_rounds'] = grouped_rounds
# metrics_df['individuals_learned'] = individuals_learned
# metrics_df['individuals_unlearned'] = individuals_unlearned


# Combine into group_df
group_df = groups_learned.merge(groups_unlearned, on=['group_id', 'domain'])


metrics_df.to_excel("corrected_combined_metrics_analysis.xlsx", index=False)
group_df.to_excel("group_data.xlsx", index=False)


In [ ]:

# Plotting function
def plot_bar_with_error(data, x, y, hue=None, title=None):
    plt.figure(figsize=(12, 6))
    sns.barplot(data=data, x=x, y=y, hue=hue, estimator=np.mean, ci=68, capsize=0.1, errwidth=1.5)
    plt.title(title or f"{y} by {x}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


In [ ]:

# Experimental condition and performance group
for var in ['total_rounds', 'total_correct']:
    plot_bar_with_error(metrics_df, x='experimental_condition', y=var, hue='performance_group',
                        title=f"{var} by Experimental Condition and Performance Group")


In [ ]:

# Domain and performance group
for var in ['total_rounds', 'total_correct']:
    plot_bar_with_error(metrics_df, x='domain', y=var, hue='performance_group',
                        title=f"{var} by Domain and Performance Group")


In [ ]:

# Rounds per KC by performance group
plot_bar_with_error(kc_round_counts_df, x='kc_id', y='count', hue='performance_group',
                    title="Average Number of Rounds per KC by Performance Group")


In [ ]:

# Optimal response ratio by performance group
kc_opt_ratio_perf = diagnostic_df.groupby(['kc_id', 'performance_group']).agg(
    avg_opt_ratio=('optimal_response_ratio', 'mean')
).reset_index()

plot_bar_with_error(kc_opt_ratio_perf, x='kc_id', y='avg_opt_ratio', hue='performance_group',
                    title="Average Optimal Response Ratio per KC by Performance Group")


In [ ]:

# Rounds by domain and performance group
sns.catplot(data=kc_round_counts_df, kind="bar", x='kc_id', y='count', hue='performance_group',
            col='domain', estimator=np.mean, ci=68, capsize=0.1, errwidth=1.5, height=6, aspect=1)
plt.subplots_adjust(top=0.85)
plt.suptitle("Rounds per KC by Domain and Performance Group")


In [ ]:

# Optimal response ratio by domain and performance group
kc_ratio_dom_perf_df = diagnostic_df.groupby(['domain', 'kc_id', 'performance_group']).agg(
    avg_opt_ratio=('optimal_response_ratio', 'mean')
).reset_index()

sns.catplot(data=kc_ratio_dom_perf_df, kind="bar", x='kc_id', y='avg_opt_ratio', hue='performance_group',
            col='domain', ci=None, height=6, aspect=1)
plt.subplots_adjust(top=0.85)
plt.suptitle("Optimal Response Ratio per KC by Domain and Performance Group")
